In [1]:
from nltk.corpus import wordnet as wn

In [2]:
import json
from datetime import datetime
import numpy as np
def jsonPretify(object_list):
    # Custom serializer to convert datetime and numpy objects
    def datetime_handler(obj):
        if isinstance(obj, datetime):
            return obj.isoformat()
        if isinstance(obj, np.ndarray):
            # Print a clean metadata summary instead of raw binary elements
            return f"<ndarray: shape={obj.shape}, dtype={obj.dtype}>"
        if isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")
        
    # Serialize using the custom handler
    pretty_json = json.dumps(object_list, indent=4, sort_keys=True, default=datetime_handler)
    return pretty_json

In [3]:


# import os
# from datasets import load_dataset

# # 1. Path to your local cache folder containing '1aurent___ade20_k'
# local_cache_dir = os.path.abspath("../datasets/ade20k")

# 2. Load the dataset (it will read from your local .arrow files without downloading again)
# dataset = load_dataset("1aurent/ADE20K", cache_dir=local_cache_dir)
# dataset = load_dataset("uva-cv-lab/ADE20k-150", cache_dir=local_cache_dir)
import os
import sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)
import expanded_benchmark_helpers as bm_hp
COCO_ANN  = os.path.join(REPO_ROOT, "datasets", "coco", "instances_val2017.json")
ADE20K_PATH = os.path.join(REPO_ROOT, "datasets", "ade20k")
PASCALVOC_PATH = os.path.join(REPO_ROOT, "datasets", "pascalvoc")
# ade_dataset = bm_hp.load_ade20k(ADE20K_PATH)
pascal_dataset = bm_hp.load_pascalvoc(PASCALVOC_PATH)




/opt/anaconda3/envs/ovseg/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded PASCAL VOC val dataset from local cache.


In [13]:
print(pascal_dataset[0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x366 at 0x173AB4B80>, 'mask': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=500x366 at 0x173A7EF10>}


In [17]:
VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle", 
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog", 
    "horse", "motorbike", "person", "pottedplant", "sheep", "sofa", 
    "train", "tvmonitor"
]
VOC_COLORMAP = [
    [0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0],
    [0, 0, 128], [128, 0, 128], [0, 128, 128], [128, 128, 128],
    [64, 0, 0], [192, 0, 0], [64, 128, 0], [192, 128, 0],
    [64, 0, 128], [192, 0, 128], [64, 128, 128], [192, 128, 128],
    [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0],
    [0, 64, 128]
]
VOC_CLS_TO_IDX = {cls : idx for idx, cls in enumerate(VOC_CLASSES)}

In [15]:
def extract_gt_objects_and_masks(dataset_item, voc_classes = VOC_CLASSES, voc_colormap = VOC_COLORMAP):
    """
    Parses a single dataset item dictionary with keys 'image' and 'mask'.
    Returns:
        gt_objects (list): List of class names present in the image.
        gt_masks (dict): Dictionary mapping class name -> binary mask (numpy array).
    """
    mask_img = dataset_item["mask"]
    
    # Convert mask PIL image to numpy array of shape (H, W, 3)
    mask_np = np.array(mask_img.convert("RGB"))
    
    gt_objects = []
    gt_masks = []
    
    # Loop through foreground classes (1 to 20)
    for class_id in range(1, len(voc_classes)):
        class_name = voc_classes[class_id]
        color = voc_colormap[class_id]
        
        # Check where the RGB channels match this class color
        binary_mask = (mask_np == color).all(axis=-1)
        
        # If the class is present in the image, keep it
        if binary_mask.any():
            gt_objects.append(class_name)
            # Store as binary mask of uint8 (0 or 1)
            gt_masks.append(binary_mask.astype(np.uint8))
            
    return gt_objects, gt_masks
# --- Example Usage ---
# Assuming item is your dataset dictionary:
# item = {'image': ..., 'mask': ...}
gt_objects, gt_masks = extract_gt_objects_and_masks(pascal_dataset[0])
print("Objects present in image:", gt_objects)
for obj_name, mask in zip(gt_objects, gt_masks):
    print(f"Mask shape for {obj_name}: {mask.shape} (Sum: {mask.sum()} pixels)")

Objects present in image: ['aeroplane']
Mask shape for aeroplane: (366, 500) (Sum: 30937 pixels)


In [10]:
print(gt_masks)

[array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8)]


In [ ]:
def get_object_size_from_mask(gt_mask):
    """
    Computes absolute area, relative area, and bounding box for each object mask.
    
    Args:
        gt_masks (dict): Dictionary mapping class_name -> binary mask (numpy array [H, W]).
    """
    object_stats = {}
    
    
    h, w = mask.shape
    total_pixels = h * w
    
    # 1. Absolute Area (sum of all 1s in binary mask)
    pixel_area = int(mask.sum())
    
    # 2. Relative Area (percentage of image)
    relative_area = (pixel_area / total_pixels) * 100
    
    # 3. Bounding Box (find coordinates where mask is True)
    y_indices, x_indices = np.argwhere(mask).T
    
    if len(x_indices) > 0:
        ymin, ymax = int(y_indices.min()), int(y_indices.max())
        xmin, xmax = int(x_indices.min()), int(x_indices.max())
        
        bbox_width = xmax - xmin + 1
        bbox_height = ymax - ymin + 1
        bbox_area = bbox_width * bbox_height
        bbox = [xmin, ymin, xmax, ymax]
    else:
        bbox = [0, 0, 0, 0]
        bbox_width = bbox_height = bbox_area = 0
        
    return {
        "pixel_area": pixel_area,
        "relative_area_pct": round(relative_area, 2),
        "bbox_coordinates_xyxy": bbox,
        "bbox_width": bbox_width,
        "bbox_height": bbox_height,
        "bbox_area": bbox_area
    }

# --- Example Usage ---
# Using `gt_masks` from the previous extraction step:
stats = get_object_size_from_mask(gt_masks[0])

print(f"For: {gt_objects[0]}")
print(f"  - Pixel Area: {stats['pixel_area']} pixels")
print(f"  - Image Coverage: {stats['relative_area_pct']}% of the image")
print(f"  - Bounding Box (W x H): {stats['bbox_width']} x {stats['bbox_height']}")
print(f"  - Bounding Box Area: {stats['bbox_area']} pixels")
print(f"  - Coordinates [xmin, ymin, xmax, ymax]: {stats['bbox_coordinates_xyxy']}")


For: aeroplane
  - Pixel Area: 30937 pixels
  - Image Coverage: 16.91% of the image
  - Bounding Box (W x H): 492 x 153
  - Bounding Box Area: 75276 pixels
  - Coordinates [xmin, ymin, xmax, ymax]: [8, 109, 499, 261]


### BenchMark

In [8]:
from sentence_transformers import SentenceTransformer, util
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device='mps') # Use 'cpu' or 'cuda' as needed

Coco

In [6]:
best_synset_id, data, definition = bm_hp.get_best_synset_id_frm_bn('person', 'person')
print(best_synset_id)
# data = bm_hp._babelnet_get_synset_data(best_synset_id)
print(data)
print(definition)

bn:16076822n
{'senses': [{'type': 'BabelSense', 'properties': {'fullLemma': 'Person_(surname)', 'simpleLemma': 'person', 'lemma': {'lemma': 'Person', 'type': 'HIGH_QUALITY'}, 'source': 'WIKI', 'senseKey': '39933035', 'frequency': 0, 'language': 'EN', 'pos': 'NOUN', 'synsetID': {'id': 'bn:16076822n', 'pos': 'NOUN', 'source': 'BABELNET'}, 'translationInfo': '', 'pronunciations': {'audios': [], 'transcriptions': []}, 'bKeySense': False, 'idSense': 101167643, 'tags': {'class it.uniroma1.lcl.babelnet.data.LabelTag': [{'language': 'EN', 'label': 'surname'}]}}}, {'type': 'BabelSense', 'properties': {'fullLemma': 'Person', 'simpleLemma': 'person', 'lemma': {'lemma': 'Person', 'type': 'HIGH_QUALITY'}, 'source': 'WIKIDATA', 'senseKey': 'Q16881132', 'frequency': 0, 'language': 'EN', 'pos': 'NOUN', 'synsetID': {'id': 'bn:16076822n', 'pos': 'NOUN', 'source': 'BABELNET'}, 'translationInfo': '', 'pronunciations': {'audios': [], 'transcriptions': []}, 'bKeySense': False, 'idSense': 496579724, 'tags': 

In [21]:
synset_ids = bm_hp._babelnet_get_synset_ids('bowl')
context_emb = embedding_model.encode('kitchen', convert_to_tensor=True)
context_txt = 'kitchen'
for sid in synset_ids[:5]:
    print(sid)
    data = bm_hp._babelnet_get_synset_data(sid)
    definition = ""
    for gloss in data.get("glosses", []):
        if gloss.get("language") == "EN":
            definition = gloss.get("gloss", "")
            break
    # candidate_text = f"{definition} related to {context_txt}"
    candidate_emb = embedding_model.encode(definition, convert_to_tensor=True)
    score = util.cos_sim(context_emb, candidate_emb).item()
    print(score)
    print(definition)
    w_s = bm_hp._babelnet_extract_english_lemmas(data)
    w_s_synset = bm_hp._babelnet_extract_english_lemmas(bm_hp._babelnet_get_synset_data(sid))
    print(w_s)
    print(w_s_synset)
    print()


bn:14620277n
0.18378393352031708
Kashkul also referred to as the beggar's bowl is a container carried by wandering Dervishes and used to collect money and other goods
['kashkul', 'shell_bowl', 'bowl', 'shell']
['kashkul', 'shell_bowl', 'bowl', 'shell']

bn:08145359n
0.2663971185684204

['bowl']
['bowl']

bn:00012500n
0.1893303096294403
The act of rolling something (as the ball in bowling)
['roll', 'bowl']
['roll', 'bowl']

bn:00012499n
0.22099699079990387
A small round container that is open at the top for holding tobacco
['bowl', 'pipe_bowl', 'marijuana_pipe']
['bowl', 'pipe_bowl', 'marijuana_pipe']

bn:00012498n
0.08300337195396423
A wooden ball (with flattened sides so that it rolls on a curved course) used in the game of lawn bowling
['bowl']
['bowl']



In [23]:
from nltk.corpus import wordnet as wn
def clean_taxonomy_list(raw_list):
    """
    Cleans a list of taxnomy words by:
      1. Splitting comma-separated strings into individual elements.
      2. Normalizing spaces to underscores and converting to lowercase.
      3. Filtering out misspelled words/typos using WordNet.
      4. Deduplicating while preserving order.
    """
    cleaned = []
    for item in raw_list:
        # Split by comma (handles "altar, communion table, Lord's table" -> ['altar', 'communion table', "Lord's table"])
        parts = [p.strip().lower() for p in item.split(",")]
        
        for part in parts:
            # Replace spaces with underscores
            part_normalized = part.replace(" ", "_")
            if not part_normalized:
                continue
                
            # Check if it's already in the cleaned list
            if part_normalized not in cleaned:
                # Use WordNet to filter out typos/misspellings
                # (e.g. "fa\u00e7ade" or "botle" will return [] and be skipped)
                if wn.synsets(part_normalized) or wn.synsets(part.replace("_", " ")):
                    cleaned.append(part_normalized)
                    
    return cleaned

In [36]:
from pycocotools import mask as maskUtils
from time import time

#command to download caoc dataset
# mkdir -p datasets/coco
# curl -L -o datasets/coco/instances_val2017.json http://images.cocodataset.org/annotations/annotations_trainval2017.zip
# unzip datasets/coco/instances_val2017.json -d datasets/coco/

def create_benchmark_from_coco(coco_dataset, limit=5000, img_bnch=[], cat_bnch={}):
    img_ids = coco_dataset.getImgIds()
    print(len(img_ids))
    for img_id in img_ids[:limit]:
        img_meta = coco_dataset.loadImgs(img_id)[0]
        h, w = img_meta["height"], img_meta["width"]

        gt_objects = []
        gt_bin_masks = []
        anns = bm_hp.get_anns_for_img_id(img_id, coco_dataset)
        
        for ann in anns:
            start_time = time()
            category = coco_dataset.loadCats(ann["category_id"])[0]
            # print(category)
            cat_name = category["name"]
            if cat_name not in gt_objects:
                gt_objects.append(cat_name)
                gt_bin_masks.append(bm_hp.get_gt_mask(coco_dataset, img_id, category['id']))
                # gt_bin_masks.append(get_bin_masks(ann["segmentation"], h, w))
            if cat_name not in cat_bnch:
                # 1. Query local WordNet first (extremely fast)
                definition_wn, w_s_wn, w_s_hp_wn, w_s_he_wn = bm_hp.build_word_sets_from_synset_v2(
                    word=cat_name, supporting_words=category["supercategory"]
                )
                definition = definition_wn
                synonyms = w_s_wn
                hyponyms = w_s_hp_wn
                hypernyms = w_s_he_wn
                
                # 2. Fallback to BabelNet ONLY if WordNet fails or has fewer than 2 synonyms
                if not w_s_wn or len(w_s_wn) < 2:
                    print(f"WordNet failed/insufficient for '{cat_name}'. Querying BabelNet API...")
                    definition_bn, w_s_bn, w_s_hp_bn, w_s_he_bn = bm_hp.supplement_word_sets_with_babelnet_v2(
                        word=cat_name, supporting_words=category["supercategory"]
                    )
                
                definition = definition_bn if definition_wn == "" else definition_wn
                synonyms.extend(w_s_bn)
                hyponyms.extend(w_s_hp_bn)
                hypernyms.extend(w_s_he_wn)
                
                
                # 3. Save to cat_bnch, limiting lists to at most 5 unique elements
                cat_bnch[cat_name].append({
                    "cat_src_id" : category["id"],
                    "cat_src" : "coco",
                    "cat_id" : f"coco_{category['id']}",
                    "definition" : definition,
                    "synonyms": clean_taxonomy_list(synonyms[:5]),
                    "hyponyms": clean_taxonomy_list(hyponyms[:5]),
                    "hypernyms": clean_taxonomy_list(hypernyms[:5])
                })
            print(f"time taken: {(time()-start_time)*1000:.1f} ms")
        
        img_bnch.append({
            "img_src" : "coco",
            "img_src_id": img_id,
            "img_id" : f"coco_{img_id}",
            "width": w,
            "height": h,
            "filename": img_meta["file_name"],
            "img_url": img_meta.get("coco_url", ""),
            "gt_objects": gt_objects,
            "gt_bin_masks": gt_bin_masks
        })

    return img_bnch, cat_bnch
    
    


In [37]:
import numpy as np
from time import time
from PIL import Image

def create_benchmark_from_ade20K(ade_dataset, limit=2000, img_bnch=[], cat_bnch={}):
    for idx in range(limit):
        print(f"Processing dataset[{idx+1}]...")
        data = ade_dataset[idx]
        img_pil = data["image"]
        img_id = int(os.path.splitext(data['filename'])[0].split('_')[-1])
        w, h = img_pil.size

        gt_objects = []
        gt_bin_masks = []
        
        for obj_id,obj in enumerate(data["objects"]):
            start_time = time()
            # print(category)
            cat_name = obj["raw_name"]
            if cat_name not in gt_objects:
                gt_objects.append(cat_name)
                gt_bin_masks.append(bm_hp.get_gt_mask_for_ade(data, cat_name))
            if cat_name not in cat_bnch:
                # 1. Query local WordNet first (extremely fast)
                hypernyms = obj["hypernym"] if len(obj["hypernym"]) > 0 else []
                definition_wn, w_s_wn, w_s_hp_wn, w_s_he_wn = bm_hp.build_word_sets_from_synset_v2(
                    word=cat_name, supporting_words=obj["hypernym"][:2]
                )
                definition = definition_wn
                synonyms = w_s_wn
                hyponyms = w_s_hp_wn
                hypernyms.extend(w_s_he_wn)
                
                # 2. Fallback to BabelNet ONLY if WordNet fails or has fewer than 2 synonyms
                if not w_s_wn:
                    print(f"WordNet failed/insufficient for '{cat_name}'. Querying BabelNet API...")
                    definition_bn, w_s_bn, w_s_hp_bn, w_s_he_bn = bm_hp.supplement_word_sets_with_babelnet_v2(
                        word=cat_name, supporting_words=obj["hypernym"][:2]
                    )
                
                    definition = definition_bn if definition_wn == "" else definition_wn
                    synonyms.extend(w_s_bn)
                    hyponyms.extend(w_s_hp_bn)
                    hypernyms.extend(w_s_he_wn)
                
                
                # 3. Save to cat_bnch, limiting lists to at most 5 unique elements
                cat_bnch[cat_name].append({
                    "cat_src_id" : obj['name_ndx'],
                    "cat_src" : "ade20k",
                    "cat_id" : f"ade20k_{obj['name_ndx']}",
                    "definition" : definition,
                    "synonyms": clean_taxonomy_list(synonyms)[:5],
                    "hyponyms": clean_taxonomy_list(hyponyms)[:5],
                    "hypernyms": clean_taxonomy_list(hypernyms)[:5]
                })
            print(f"time taken: {(time()-start_time)*1000:.1f} ms")
        
        img_bnch.append({
            "img_src" : "ade20k",
            "img_src_id": img_id,
            "img_id" : f"ade20k_{img_id}",
            "width": w,
            "height": h,
            "filename": data["filename"],
            "img_url": "",
            "gt_objects": gt_objects,
            "gt_bin_masks": gt_bin_masks
        })
        print("Processing completed!!")

    return img_bnch, cat_bnch

In [24]:
from time import time

def create_benchmark_from_pascalvoc(pascal_dataset, limit, img_bnch=[], cat_bnch={}):
    for idx in range(limit):
        print(f"Processing dataset[{idx+1}]...")
        data = pascal_dataset[idx]
        img_pil = data["image"]
        img_id = idx
        w, h = img_pil.size

        gt_objects, gt_bin_masks = extract_gt_objects_and_masks(data)
        
        for obj_id,obj in enumerate(gt_objects):
            start_time = time()
            # print(category)
            cat_name = obj
            # if cat_name not in gt_objects:
            #     gt_objects.append(cat_name)
            #     gt_bin_masks.append(bm_hp.get_gt_mask_for_ade(data, cat_name))
            if cat_name not in cat_bnch:
                # 1. Query local WordNet first (extremely fast)
                definition_wn, w_s_wn, w_s_hp_wn, w_s_he_wn = bm_hp.build_word_sets_from_synset_v2(
                    word=cat_name, supporting_words=obj
                )
                definition = definition_wn
                synonyms = w_s_wn
                hyponyms = w_s_hp_wn
                hypernyms = w_s_he_wn
                
                # 2. Fallback to BabelNet ONLY if WordNet fails or has fewer than 2 synonyms
                if not w_s_wn:
                    print(f"WordNet failed/insufficient for '{cat_name}'. Querying BabelNet API...")
                    definition_bn, w_s_bn, w_s_hp_bn, w_s_he_bn = bm_hp.supplement_word_sets_with_babelnet_v2(
                        word=cat_name, supporting_words=obj
                    )
                
                    definition = definition_bn if definition_wn == "" else definition_wn
                    synonyms.extend(w_s_bn)
                    hyponyms.extend(w_s_hp_bn)
                    hypernyms.extend(w_s_he_wn)
                
                
                # 3. Save to cat_bnch, limiting lists to at most 5 unique elements
                cat_bnch[cat_name].append({
                    "cat_src_id" : VOC_CLS_TO_IDX[obj],
                    "cat_src" : "pascalvoc",
                    "cat_id" : f"pascalvoc_{VOC_CLS_TO_IDX[obj]}",
                    "definition" : definition,
                    "synonyms": clean_taxonomy_list(synonyms)[:5],
                    "hyponyms": clean_taxonomy_list(hyponyms)[:5],
                    "hypernyms": clean_taxonomy_list(hypernyms)[:5]
                })
            print(f"time taken: {(time()-start_time)*1000:.1f} ms")
        
        img_bnch.append({
            "img_src" : "pascalvoc",
            "img_src_id": img_id,
            "img_id" : f"pascalvoc_{img_id}",
            "width": w,
            "height": h,
            "filename": "",
            "img_url": "",
            "gt_objects": gt_objects,
            "gt_bin_masks": gt_bin_masks
        })
        print("Processing completed!!")

    return img_bnch, cat_bnch

In [30]:
from collections import defaultdict
img_bnch = []
cat_bnch = defaultdict(list)
img_c, cat_c = create_benchmark_from_pascalvoc(pascal_dataset, 5, img_bnch, cat_bnch)
# img, cat = create_benchmark_from_ade20K(ade_dataset, 5, img_c, cat_c)
# img, cat = create_benchmark_from_ade20K(ade_dataset, 5, img_c, cat_c)

Processing dataset[1]...
time taken: 3.1 ms
Processing completed!!
Processing dataset[2]...
time taken: 960.3 ms
Processing completed!!
Processing dataset[3]...
time taken: 94.0 ms
Processing completed!!
Processing dataset[4]...
time taken: 0.0 ms
Processing completed!!
Processing dataset[5]...
time taken: 6.1 ms
time taken: 211.1 ms
Processing completed!!


In [31]:
print(jsonPretify(img_c))

[
    {
        "filename": "",
        "gt_bin_masks": [
            "<ndarray: shape=(366, 500), dtype=uint8>"
        ],
        "gt_objects": [
            "aeroplane"
        ],
        "height": 366,
        "img_id": "pascalvoc_0",
        "img_src": "pascalvoc",
        "img_src_id": 0,
        "img_url": "",
        "width": 500
    },
    {
        "filename": "",
        "gt_bin_masks": [
            "<ndarray: shape=(335, 500), dtype=uint8>"
        ],
        "gt_objects": [
            "train"
        ],
        "height": 335,
        "img_id": "pascalvoc_1",
        "img_src": "pascalvoc",
        "img_src_id": 1,
        "img_url": "",
        "width": 500
    },
    {
        "filename": "",
        "gt_bin_masks": [
            "<ndarray: shape=(333, 500), dtype=uint8>"
        ],
        "gt_objects": [
            "boat"
        ],
        "height": 333,
        "img_id": "pascalvoc_2",
        "img_src": "pascalvoc",
        "img_src_id": 2,
        "img_url": "",


In [32]:
print(jsonPretify(cat_c))

{
    "aeroplane": [
        {
            "cat_id": "pascalvoc_1",
            "cat_src": "pascalvoc",
            "cat_src_id": 1,
            "definition": "an aircraft that has a fixed wing and is powered by propellers or jets",
            "hypernyms": [
                "heavier-than-air_craft"
            ],
            "hyponyms": [
                "hangar_queen",
                "monoplane",
                "propeller_plane",
                "seaplane",
                "hydroplane"
            ],
            "synonyms": [
                "airplane",
                "aeroplane",
                "plane"
            ]
        }
    ],
    "bicycle": [
        {
            "cat_id": "pascalvoc_2",
            "cat_src": "pascalvoc",
            "cat_src_id": 2,
            "definition": "a wheeled vehicle that has two wheels and is moved by foot pedals",
            "hypernyms": [
                "wheeled_vehicle"
            ],
            "hyponyms": [
                "safety_bi

In [30]:
from collections import defaultdict
def build_pos_neg_sets(img_bnch):
    """
    Builds positive and negative image sets for categories in the benchmark.
    
    Returns:
      positive_set: {category: [img_ids where category exists]}
      negative_set: {category: [img_ids where category does NOT exist]}
    """
    # 1. Collect all unique image IDs in this benchmark
    all_img_ids = [img["img_id"] for img in img_bnch]
    all_img_ids_set = set(all_img_ids)
    
    # 2. Build the positive set
    positive_set = defaultdict(list)
    for img in img_bnch:
        img_id = img["img_id"]
        for cat in img["gt_objects"]:
            if img_id not in positive_set[cat]:
                positive_set[cat].append(img_id)
                
    # Sort positive lists for consistency
    positive_set = {cat: sorted(ids) for cat, ids in positive_set.items()}
    
    # 3. Build the negative set (All IDs minus positive IDs)
    negative_set = {}
    for cat, pos_ids in positive_set.items():
        pos_set = set(pos_ids)
        negative_set[cat] = sorted(list(all_img_ids_set - pos_set))
        
    return positive_set, negative_set

In [42]:
# Combine both benchmarks into a single list

pos_combined, neg_combined = build_pos_neg_sets(img)

print(jsonPretify(pos_combined))


{
    "air conditioning": [
        "ade20k_1029"
    ],
    "altar": [
        "ade20k_25"
    ],
    "aquarium": [
        "ade20k_26"
    ],
    "balustrade": [
        "ade20k_25"
    ],
    "banana": [
        "coco_37777"
    ],
    "bicycle": [
        "coco_174482",
        "coco_87038"
    ],
    "book": [
        "ade20k_1029"
    ],
    "bookcase": [
        "ade20k_29"
    ],
    "books": [
        "ade20k_29"
    ],
    "bottle": [
        "coco_397133"
    ],
    "bowl": [
        "coco_397133"
    ],
    "box": [
        "ade20k_1029"
    ],
    "boxes": [
        "ade20k_29"
    ],
    "broccoli": [
        "coco_397133"
    ],
    "button": [
        "ade20k_1028"
    ],
    "button panel": [
        "ade20k_1028"
    ],
    "buttons": [
        "ade20k_1028"
    ],
    "cabinet": [
        "ade20k_1028",
        "ade20k_1029"
    ],
    "car": [
        "coco_174482"
    ],
    "carrot": [
        "coco_397133"
    ],
    "ceiling": [
        "ade20k_1028",
        "a

In [43]:
print(jsonPretify(neg_combined))


{
    "air conditioning": [
        "ade20k_1028",
        "ade20k_25",
        "ade20k_26",
        "ade20k_29",
        "coco_174482",
        "coco_252219",
        "coco_37777",
        "coco_397133",
        "coco_87038"
    ],
    "altar": [
        "ade20k_1028",
        "ade20k_1029",
        "ade20k_26",
        "ade20k_29",
        "coco_174482",
        "coco_252219",
        "coco_37777",
        "coco_397133",
        "coco_87038"
    ],
    "aquarium": [
        "ade20k_1028",
        "ade20k_1029",
        "ade20k_25",
        "ade20k_29",
        "coco_174482",
        "coco_252219",
        "coco_37777",
        "coco_397133",
        "coco_87038"
    ],
    "balustrade": [
        "ade20k_1028",
        "ade20k_1029",
        "ade20k_26",
        "ade20k_29",
        "coco_174482",
        "coco_252219",
        "coco_37777",
        "coco_397133",
        "coco_87038"
    ],
    "banana": [
        "ade20k_1028",
        "ade20k_1029",
        "ade20k_25",
        "a

In [8]:
import pickle
import json
with open("../data/img_metadata.pkl", "rb") as f:
        img_bnch = pickle.load(f)
print(f"Loaded {len(img_bnch)} cached images.")

with open("../data/word_sets_v2.json", "rb") as f:
        cat_bnch = json.load(f)
print(f"Loaded {len(cat_bnch)} cached categories.")

Loaded 22357 cached images.
Loaded 346 cached categories.


In [16]:
print(set([img["img_src"] for img in img_bnch]))

{'pascalvoc', 'lvis', 'ade20k'}


In [9]:
filtered_list = []
for d in img_bnch[:5]:
    d_copy = d.copy()            # Copy the dictionary to keep the original untouched
    d_copy.pop("gt_bin_masks", None)  # Remove 'gt_bin_masks' if it exists
    filtered_list.append(d_copy)

print(jsonPretify(filtered_list))

[
    {
        "filename": "000000397133.jpg",
        "gt_objects": [
            "bottle",
            "dining table",
            "person",
            "knife",
            "bowl",
            "oven",
            "cup",
            "broccoli",
            "spoon",
            "carrot",
            "sink"
        ],
        "height": 427,
        "img_id": "coco_397133",
        "img_src": "coco",
        "img_src_id": 397133,
        "img_url": "http://images.cocodataset.org/val2017/000000397133.jpg",
        "width": 640
    },
    {
        "filename": "000000037777.jpg",
        "gt_objects": [
            "potted plant",
            "chair",
            "dining table",
            "refrigerator",
            "banana",
            "oven",
            "sink",
            "orange"
        ],
        "height": 230,
        "img_id": "coco_37777",
        "img_src": "coco",
        "img_src_id": 37777,
        "img_url": "http://images.cocodataset.org/val2017/000000037777.jpg",
    

In [27]:
import zlib
img_info = img_bnch[0]
img_id = img_info["img_src_id"]
for idx, (obj, obj_mask) in enumerate(zip(img_info["gt_objects"],img_info["gt_bin_masks"])):

    mask = np.frombuffer(zlib.decompress(obj_mask), dtype=np.uint8).reshape((img_info["height"], img_info["width"]))
    stats = get_object_size_from_mask(mask)

    print(f"\nStats for {obj.upper()}:")
    print(f"  - Pixel Area: {stats['pixel_area']} pixels")
    print(f"  - Image Coverage: {stats['relative_area_pct']}% of the image")
    print(f"  - Bounding Box (W x H): {stats['bbox_width']} x {stats['bbox_height']}")
    print(f"  - Bounding Box Area: {stats['bbox_area']} pixels")
    print(f"  - Coordinates [xmin, ymin, xmax, ymax]: {stats['bbox_coordinates_xyxy']}")


Stats for BOTTLE:
  - Pixel Area: 1482 pixels
  - Image Coverage: 0.54% of the image
  - Bounding Box (W x H): 39 x 57
  - Bounding Box Area: 2223 pixels
  - Coordinates [xmin, ymin, xmax, ymax]: [218, 241, 256, 297]

Stats for DINING TABLE:
  - Pixel Area: 54088 pixels
  - Image Coverage: 19.79% of the image
  - Bounding Box (W x H): 347 x 187
  - Bounding Box Area: 64889 pixels
  - Coordinates [xmin, ymin, xmax, ymax]: [1, 240, 347, 426]

Stats for PERSON:
  - Pixel Area: 18463 pixels
  - Image Coverage: 6.76% of the image
  - Bounding Box (W x H): 498 x 277
  - Bounding Box Area: 137946 pixels
  - Coordinates [xmin, ymin, xmax, ymax]: [0, 70, 497, 346]

Stats for KNIFE:
  - Pixel Area: 128 pixels
  - Image Coverage: 0.05% of the image
  - Bounding Box (W x H): 21 x 29
  - Bounding Box Area: 609 pixels
  - Coordinates [xmin, ymin, xmax, ymax]: [136, 249, 156, 277]

Stats for BOWL:
  - Pixel Area: 4716 pixels
  - Image Coverage: 1.73% of the image
  - Bounding Box (W x H): 151 x 271


In [ ]:
import numpy as np
from PIL import Image
from transformers.image_utils import load_image
from pycocotools.mask import frPyObjects, decode


def load_image_lazy(img_info, dataset= None):
    """
    Loads PIL Image on-the-fly without saving local image files.
    """
    if img_info["img_src"] == "coco":
        # Load directly from public COCO URL into RAM
        return load_image(img_info["img_url"])
    elif img_info["img_src"] == "ade20k":
        # Pull from the memory-mapped HF dataset
        if dataset is None:
            raise ValueError("ade_dataset must be provided to resolve ADE20K images")
        return dataset[img_info["img_src_id"]]["image"]
    raise ValueError("Unknown image source")

img = load_image_lazy(img_info)
img.show()

In [7]:
img = pascal_dataset[0]['image']
img.show()

In [33]:
import os
import json
import zipfile
import requests
REPO_ROOT = "/Users/ushnesha/NextDrive/Documents/Academic/OVS_Eval/taxonomy_driven_ovs_models"
LVIS_DIR = os.path.join(REPO_ROOT, "datasets", "lvis")
os.makedirs(LVIS_DIR, exist_ok=True)
LVIS_ZIP_PATH = os.path.join(LVIS_DIR, "lvis_v1_val.json.zip")
LVIS_JSON_PATH = os.path.join(LVIS_DIR, "lvis_v1_val.json")
# Step 1: Download LVIS annotations zip if not present
if not os.path.exists(LVIS_JSON_PATH):
    if not os.path.exists(LVIS_ZIP_PATH):
        url = "https://dl.fbaipublicfiles.com/LVIS/lvis_v1_val.json.zip"
        print(f"Downloading LVIS v1.0 validation annotations from {url}...")
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(LVIS_ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Download complete.")
    # Step 2: Extract zip
    print(f"Extracting {LVIS_ZIP_PATH}...")
    with zipfile.ZipFile(LVIS_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LVIS_DIR)
    print("Extraction complete.")
# Step 3: Load and inspect JSON
print(f"Loading {LVIS_JSON_PATH}...")
with open(LVIS_JSON_PATH, 'r') as f:
    data = json.load(f)
print("\nLVIS JSON Structure Keys:", list(data.keys()))
print(f"Number of categories: {len(data['categories'])}")
print(f"Number of images: {len(data['images'])}")
print(f"Number of annotations: {len(data['annotations'])}")
print("\nFirst Category Example:")
print(json.dumps(data['categories'][0], indent=2))
print("\nFirst Image Example:")
print(json.dumps(data['images'][0], indent=2))
print("\nFirst Annotation Example:")
ann_example = data['annotations'][0]
# Omit 'segmentation' from print as it's huge
print(json.dumps({k: v for k, v in ann_example.items() if k != 'segmentation'}, indent=2))

Download complete.
Extracting /Users/ushnesha/NextDrive/Documents/Academic/OVS_Eval/taxonomy_driven_ovs_models/datasets/lvis/lvis_v1_val.json.zip...
Extraction complete.
Loading /Users/ushnesha/NextDrive/Documents/Academic/OVS_Eval/taxonomy_driven_ovs_models/datasets/lvis/lvis_v1_val.json...

LVIS JSON Structure Keys: ['info', 'categories', 'annotations', 'images', 'licenses']
Number of categories: 1203
Number of images: 19809
Number of annotations: 244707

First Category Example:
{
  "image_count": 8,
  "synonyms": [
    "aerosol_can",
    "spray_can"
  ],
  "def": "a dispenser that holds a substance under pressure",
  "id": 1,
  "synset": "aerosol.n.02",
  "name": "aerosol_can",
  "frequency": "c",
  "instance_count": 11
}

First Image Example:
{
  "date_captured": "2013-11-14 17:02:52",
  "neg_category_ids": [
    279,
    899,
    127,
    180,
    1136,
    725,
    663
  ],
  "id": 397133,
  "license": 4,
  "height": 427,
  "width": 640,
  "flickr_url": "http://farm7.staticflickr

In [37]:
print(data['categories'][0])

{'image_count': 8, 'synonyms': ['aerosol_can', 'spray_can'], 'def': 'a dispenser that holds a substance under pressure', 'id': 1, 'synset': 'aerosol.n.02', 'name': 'aerosol_can', 'frequency': 'c', 'instance_count': 11}


In [38]:
print(data['images'][0])

{'date_captured': '2013-11-14 17:02:52', 'neg_category_ids': [279, 899, 127, 180, 1136, 725, 663], 'id': 397133, 'license': 4, 'height': 427, 'width': 640, 'flickr_url': 'http://farm7.staticflickr.com/6116/6255196340_da26cf2c9e_z.jpg', 'coco_url': 'http://images.cocodataset.org/val2017/000000397133.jpg', 'not_exhaustive_category_ids': [914, 801, 566, 139, 1021]}


In [39]:
print(data['annotations'][0])

{'area': 73297.48, 'id': 1, 'segmentation': [[270.75, 598.57, 261.98, 598.57, 247.84, 598.57, 234.68, 598.08, 221.52, 598.08, 208.84, 598.08, 196.17, 598.08, 185.93, 597.59, 173.26, 597.1, 163.02, 597.1, 152.79, 597.59, 143.04, 597.59, 136.21, 598.57, 134.26, 605.88, 127.93, 611.24, 123.05, 617.58, 122.56, 625.38, 125.0, 640.0, 110.87, 640.0, 108.43, 640.0, 103.07, 626.35, 99.17, 621.48, 99.17, 613.19, 100.63, 603.44, 103.55, 597.59, 102.58, 588.33, 101.6, 576.14, 101.6, 563.47, 99.17, 551.28, 96.73, 535.2, 97.22, 523.01, 100.14, 512.77, 102.09, 507.41, 105.02, 504.98, 103.55, 492.3, 101.6, 478.65, 99.65, 468.42, 91.85, 463.05, 86.01, 455.26, 83.08, 446.48, 86.49, 438.68, 91.37, 427.47, 96.73, 421.13, 101.6, 413.33, 107.45, 405.05, 118.66, 395.79, 124.51, 387.5, 128.41, 382.62, 134.75, 380.67, 132.31, 368.98, 131.83, 357.76, 131.83, 345.58, 129.88, 332.42, 127.44, 318.77, 124.51, 294.88, 125.49, 282.7, 123.54, 267.1, 125.0, 257.35, 126.75, 249.45, 131.1, 241.63, 137.19, 236.41, 144.14,

In [42]:
cat_id_to_meta = {cat["id"]: cat for cat in data["categories"][:5]}
print(jsonPretify(cat_id_to_meta))
# for idx in range(5):


{
    "1": {
        "def": "a dispenser that holds a substance under pressure",
        "frequency": "c",
        "id": 1,
        "image_count": 8,
        "instance_count": 11,
        "name": "aerosol_can",
        "synonyms": [
            "aerosol_can",
            "spray_can"
        ],
        "synset": "aerosol.n.02"
    },
    "2": {
        "def": "a machine that keeps air cool and dry",
        "frequency": "f",
        "id": 2,
        "image_count": 65,
        "instance_count": 146,
        "name": "air_conditioner",
        "synonyms": [
            "air_conditioner"
        ],
        "synset": "air_conditioner.n.01"
    },
    "3": {
        "def": "an aircraft that has a fixed wing and is powered by propellers or jets",
        "frequency": "f",
        "id": 3,
        "image_count": 349,
        "instance_count": 619,
        "name": "airplane",
        "synonyms": [
            "airplane",
            "aeroplane"
        ],
        "synset": "airplane.n.01"
    },

In [43]:
target_images = data["images"][:5]
print(jsonPretify(target_images))

[
    {
        "coco_url": "http://images.cocodataset.org/val2017/000000397133.jpg",
        "date_captured": "2013-11-14 17:02:52",
        "flickr_url": "http://farm7.staticflickr.com/6116/6255196340_da26cf2c9e_z.jpg",
        "height": 427,
        "id": 397133,
        "license": 4,
        "neg_category_ids": [
            279,
            899,
            127,
            180,
            1136,
            725,
            663
        ],
        "not_exhaustive_category_ids": [
            914,
            801,
            566,
            139,
            1021
        ],
        "width": 640
    },
    {
        "coco_url": "http://images.cocodataset.org/val2017/000000037777.jpg",
        "date_captured": "2013-11-14 20:55:31",
        "flickr_url": "http://farm9.staticflickr.com/8429/7839199426_f6d48aa585_z.jpg",
        "height": 230,
        "id": 37777,
        "license": 1,
        "neg_category_ids": [
            1002,
            434,
            924,
            928,
 

In [44]:
img_ids = {img["id"] for img in target_images}
print(img_ids)

{397133, 37777, 174482, 252219, 87038}


In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
def group_words_by_embeddings(words, distance_threshold=0.4):
    # Load embedding model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(words)
    # Hierarchical / Cosine distance clustering
    clustering = AgglomerativeClustering(
        n_clusters=None, 
        distance_threshold=distance_threshold, 
        metric='cosine', 
        linkage='average'
    )
    clustering.fit(embeddings)
    # Group words by cluster ID
    clusters = {}
    for word, label in zip(words, clustering.labels_):
        clusters.setdefault(int(label), []).append(word)
    return list(clusters.values())
# Example:
words = list(cat_bnch.keys())
grouped_clusters = group_words_by_embeddings(words, distance_threshold=0.2)
for i, cluster in enumerate(grouped_clusters, 1):
    print(f"Group {i}: {cluster}")

Group 1: ['Bannister']
Group 2: ['Christmas bush']
Group 3: ['Frisbee']
Group 4: ['New Jersey']
Group 5: ['Ottoman']
Group 6: ['air conditioner']
Group 7: ['airplane']
Group 8: ['apple']
Group 9: ['apron']
Group 10: ['arm']
Group 11: ['armchair']
Group 12: ['ashcan']
Group 13: ['awning']
Group 14: ['baby buggy']
Group 15: ['back']
Group 16: ['back pillow', 'pillow']
Group 17: ['backpack']
Group 18: ['bag']
Group 19: ['balcony']
Group 20: ['ball', 'soccer ball']
Group 21: ['banana']
Group 22: ['banner']
Group 23: ['base']
Group 24: ['baseball']
Group 25: ['baseball bat']
Group 26: ['baseball cap']
Group 27: ['baseball glove']
Group 28: ['basket']
Group 29: ['bath mat']
Group 30: ['bath towel', 'towel']
Group 31: ['bathtub']
Group 32: ['beanie']
Group 33: ['bear']
Group 34: ['bed']
Group 35: ['beer bottle']
Group 36: ['belt']
Group 37: ['bench']
Group 38: ['bicycle']
Group 39: ['bird']
Group 40: ['blanket']
Group 41: ['blender']
Group 42: ['blind']
Group 43: ['blouse']
Group 44: ['boat']